# Course Design Demo

本 Notebook 展示“面向本地生活商户的商品级广告增长决策 Agent”。项目聚焦数字经济/平台经济中的本地生活商户增长问题：商户希望提升曝光、点击、转化和订单；POI 级广告难以精确承接高意向搜索；商品级广告需要解决主推品选择、Query-SKU 匹配、出价和 ROI 风险控制。

所有数据均为 synthetic demo data，不包含任何公司内部数据。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'app').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from app.agent.graph import run_agent
from app.db.init_db import initialize_database

initialize_database()
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR

## 数据介绍

读取 products、orders、traffic、reviews、campaigns、local_ad_sku_candidates、query_sku_recall、ad_bid_experiments 和 poi_level_ads_baseline。

In [ ]:
tables = {
    name: pd.read_csv(DATA_DIR / f'{name}.csv')
    for name in [
        'products',
        'orders',
        'traffic',
        'reviews',
        'campaigns',
        'local_ad_sku_candidates',
        'query_sku_recall',
        'ad_bid_experiments',
        'poi_level_ads_baseline',
    ]
}
[(name, df.shape) for name, df in tables.items()]

In [ ]:
tables['local_ad_sku_candidates'].head()

## 经营归因 Demo

In [ ]:
business_result = run_agent('商品 P1001 最近 GMV 为什么下降？')
business_result['intent'], business_result['trace_id'], business_result['tool_results'].keys()

In [ ]:
print(business_result['final_answer'][:1800])

## 主推品挖掘 Demo

In [ ]:
sku_result = run_agent('商户 M001 应该优先推哪些商品做搜索广告？')
sku_result['intent'], sku_result['tool_results']['product_ad']['sku_mining']['candidates'][:3]

## 出价与 ROI 守护 Demo

In [ ]:
bid_range_result = run_agent('P1001 如果做主推品加价，合理出价区间是多少？')
bid_roi_result = run_agent('P1001 加价 20% 投放后 ROI 是否还有保障？')
bid_range_result['tool_results']['product_ad']['bid_range'], bid_roi_result['tool_results']['product_ad']['bid_simulation']

## Query-SKU 召回 Demo

In [ ]:
recall_result = run_agent('用户搜索 水光补水 时，应该召回哪些商品，为什么？')
recall_result['tool_results']['product_ad']['query_recall']['results']

## POI 级广告 vs 商品级广告 Demo

In [ ]:
compare_result = run_agent('商户 M001 从门店级广告升级到商品级广告，CTR、CVR 和 ROI 有什么变化？')
pd.DataFrame(compare_result['tool_results']['product_ad']['comparison']['comparison'])

## Eval 实验

In [ ]:
from evals.run_eval import run_evaluations

eval_result = run_evaluations()
eval_result['case_count'], eval_result['overall_metrics']

## Ablation 实验

In [ ]:
from evals.run_ablation import run_ablation

ablation_result = run_ablation()
ablation_result['overall_metrics_by_mode']

## 可视化

In [ ]:
overall = eval_result['overall_metrics']
metric_names = ['intent_accuracy', 'avg_bid_guardrail', 'avg_sku_recall_fields', 'avg_score']
plt.figure(figsize=(8, 4))
plt.bar(metric_names, [overall[name] for name in metric_names])
plt.xticks(rotation=20)
plt.title('Eval Metrics')
plt.ylim(0, 1)
plt.show()

ablation_df = pd.DataFrame(ablation_result['overall_metrics_by_mode']).T
plt.figure(figsize=(8, 4))
plt.bar(ablation_df.index, ablation_df['avg_score'])
plt.xticks(rotation=30, ha='right')
plt.title('Ablation Avg Score')
plt.ylim(0, 1)
plt.show()

sku_df = tables['local_ad_sku_candidates'].query("merchant_id == 'M001'").copy()
sku_df['simple_growth_view'] = sku_df['cvr'] + sku_df['gmv_share'] + sku_df['pcvr']
plt.figure(figsize=(7, 4))
plt.bar(sku_df['product_id'], sku_df['simple_growth_view'])
plt.title('M001 Product Growth Signal')
plt.show()

bid_df = tables['ad_bid_experiments'].query("product_id == 'P1001'")
plt.figure(figsize=(7, 4))
plt.plot(bid_df['bid_multiplier'], bid_df['roi'], marker='o')
plt.title('Bid Multiplier vs ROI')
plt.xlabel('bid_multiplier')
plt.ylabel('roi')
plt.show()

compare_df = tables['poi_level_ads_baseline'].query("merchant_id == 'M001'")
compare_df.set_index('campaign_type')[['ctr', 'cvr', 'roi']].plot(kind='bar', figsize=(8, 4))
plt.title('POI-level vs Product-level Ads')
plt.xticks(rotation=20, ha='right')
plt.show()